# Nutrition5k: VLMs por API (zero-shot, en local)

Referencia, no candidatos para el móvil: Gemini 3.8 Flash y DeepSeek Flash (baratos) y Gemini 4 Argon (caro), elegidos
el 2026-11-02 (`notes/decisions_and_concepts/reference-models.md`). Cada modelo recibe el recorte del plato a 448 px
y el mismo prompt (`PROMPT` en `src/nutrition5k/vlm.py`), con su muestreo y razonamiento por defecto, y nunca se
entrena con Nutrition5k. Las respuestas se guardan en `outputs/vlm/full/<run>/responses.jsonl`; un run interrumpido
continúa con los platos que faltan.

- Claves: variables de entorno `GEMINI_API_KEY` y `DEEPSEEK_API_KEY` (nunca en el repositorio).
- Paso 1: 30 platos de validación por modelo, para comprobar que las respuestas se leen y que los valores tienen
  sentido. Si hay que cambiar el prompt, se cambia aquí, antes del test, y se repite el paso 1 (borrando esos runs).
- Paso 2: los 507 platos de test. Coste estimado de todo: < 20 $ (ver `vlm_costs` en `results_general.ipynb`).


In [ ]:
import os
import sys
from pathlib import Path

if Path.cwd().parent.name == "notebooks":
    os.chdir("../..")
PYTHON = sys.executable
MODELS = [("deepseek", "deepseek-flash"), ("gemini", "gemini-3.8-flash"), ("gemini", "gemini-4-argon")]
for variable in ("GEMINI_API_KEY", "DEEPSEEK_API_KEY"):
    print(variable, "set" if os.environ.get(variable) else "MISSING")


def vlm(model, provider, split, limit=None, workers=8):
    """One VLM run of `python -m nutrition5k vlm`; finished dishes are skipped."""
    command = f"{PYTHON} -m nutrition5k vlm --model {model} --provider {provider} --split {split} --workers {workers} --full"
    if limit:
        command += f" --limit {limit}"
    print(command)
    get_ipython().system(command)
    if get_ipython().user_ns["_exit_code"]:
        raise RuntimeError(f"vlm failed: {command}")


## Paso 1: comprobar el prompt (30 platos de validación)

In [ ]:
for provider, model in MODELS:
    vlm(model, provider, "val", limit=30)

from nutrition5k.results import load_vlm

checks = load_vlm()
checks[checks["split"] == "val"][["model", "dishes", "parse_failures", "val_mean_pmae", "prompt_tokens",
                                  "output_tokens", "seconds"]].round(2)


## Paso 2: test (507 platos)

In [ ]:
for provider, model in MODELS:
    vlm(model, provider, "test")
